# IPD — CCTV Gender Analytics Pipeline (v9.3, no age)

Self-contained Colab notebook. Runs the full pipeline:

1. YOLOv8n person detector (with aspect-ratio filter)
2. Containment-aware dedup (merges small-box-inside-big-box)
3. YuNet face detector (v6 score bug fixed — reads `row[14]`)
4. Landmark geometry validation
5. Face ViT gender (`rizvandwiki/gender-classification-2`)
6. CLIP zero-shot body gender (`openai/clip-vit-base-patch32`)
7. Ensemble decision tree with **v9.2 'uncertain' branch**
8. IoU + size-scaled centroid Tracker (persist=3)
9. **v9.3 ReIDBank** (CLIP cosine ≥ 0.75 reuse)
10. Line crossing footfall counter
11. 15-min bucket aggregator
12. Report generator (CSV + PNG + JSON)

**Age is intentionally NOT classified in this version.**


In [ ]:
# === Install dependencies (Colab) ===
# Run once per Colab session.
!pip install -q opencv-python-headless opencv-contrib-python-headless \
                ultralytics transformers pillow pyyaml matplotlib
# CPU torch (much smaller than default GPU build)
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cpu

# Enable GPU in Colab: Runtime > Change runtime type > T4 GPU
# Then change DEVICE = 'cuda' in the Config cell below.


In [ ]:
# === Imports ===
from __future__ import annotations
import os, sys, time, json, math, urllib.request
from dataclasses import dataclass, field
from collections import defaultdict, deque
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import cv2
import numpy as np
import yaml

import torch
from PIL import Image
from transformers import (
    AutoImageProcessor, AutoModelForImageClassification,
    CLIPModel, CLIPProcessor,
)
from ultralytics import YOLO

print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())
print('cv2', cv2.__version__)


## Config

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

CONFIG = {
    'detector': {
        'yolo_model': 'yolov8n.pt',
        'conf': 0.20,
        'device': DEVICE,
        'imgsz': None,
        'min_box_w': 20, 'min_box_h': 30,
        'aspect_min': 0.15, 'aspect_max': 6.0,
    },
    'face_detector': {
        'model_path': 'models/face_detection_yunet_2023mar.onnx',
        'conf_threshold': 0.45,
        'nms_threshold': 0.30,
        'top_k': 5000,
    },
    'face_gender': {
        'model': 'rizvandwiki/gender-classification-2',
        'female_label': 'female', 'male_label': 'male',
        'conf_threshold': 0.70,
    },
    'clip': {
        'model': 'openai/clip-vit-base-patch32',
        'gender_male_prompts': ['a photo of a man', 'a photo of a male person'],
        'gender_female_prompts': ['a photo of a woman', 'a photo of a female person'],
        'uncertain_ratio': 1.8,
        'uncertain_face_threshold': 0.70,
    },
    'tracker': {
        'iou_threshold': 0.30,
        'centroid_max_dist': 80.0,
        'persist_frames': 3,
        'size_scale': 0.30,
    },
    'reid': {
        'enabled': True,
        'cosine_threshold': 0.75,
        'expiry_seconds': 30.0,
        'max_bank_size': 200,
    },
    'footfall': {
        'line': [[0.0, 0.7], [1.0, 0.7]],
        'direction': 'down',
    },
    'aggregator': {'bucket_seconds': 900},
    'pipeline': {
        'process_every_n_frames': 1,
        'dump_annotated_every': 30,
    },
}


## Geometry utilities

In [ ]:
def iou(a, b):
    xa, ya = max(float(a[0]), float(b[0])), max(float(a[1]), float(b[1]))
    xb, yb = min(float(a[2]), float(b[2])), min(float(a[3]), float(b[3]))
    if xb <= xa or yb <= ya: return 0.0
    inter = (xb - xa) * (yb - ya)
    aa = max(0.0, float(a[2])-float(a[0])) * max(0.0, float(a[3])-float(a[1]))
    ab = max(0.0, float(b[2])-float(b[0])) * max(0.0, float(b[3])-float(b[1]))
    return inter / (aa + ab - inter) if (aa + ab - inter) > 0 else 0.0

def contains(outer, inner, frac=0.85):
    xa, ya = max(float(outer[0]), float(inner[0])), max(float(outer[1]), float(inner[1]))
    xb, yb = min(float(outer[2]), float(inner[2])), min(float(outer[3]), float(inner[3]))
    if xb <= xa or yb <= ya: return False
    inter = (xb - xa) * (yb - ya)
    inner_area = max(1.0, (float(inner[2])-float(inner[0])) * (float(inner[3])-float(inner[1])))
    return (inter / inner_area) >= frac

def dedup_boxes(boxes, scores, iou_thresh=0.45, contain_thresh=0.85):
    if len(boxes) == 0:
        return np.zeros((0,4), dtype=np.float32), np.zeros((0,), dtype=np.float32)
    boxes = np.asarray(boxes, dtype=np.float32)
    scores = np.asarray(scores, dtype=np.float32)
    order = np.argsort(-scores)
    keep, suppressed = [], np.zeros(len(boxes), dtype=bool)
    for i in order:
        if suppressed[i]: continue
        keep.append(int(i))
        for j in order:
            if j == i or suppressed[j]: continue
            iou_ij = iou(boxes[i], boxes[j])
            ai = (float(boxes[i][2])-float(boxes[i][0])) * (float(boxes[i][3])-float(boxes[i][1]))
            aj = (float(boxes[j][2])-float(boxes[j][0])) * (float(boxes[j][3])-float(boxes[j][1]))
            if ai >= aj:
                if contains(boxes[i], boxes[j], contain_thresh):
                    suppressed[j] = True; continue
            else:
                if contains(boxes[j], boxes[i], contain_thresh):
                    suppressed[i] = True; keep.pop(); keep.append(int(j)); break
            if iou_ij >= iou_thresh:
                if scores[j] < scores[i] or (scores[j] == scores[i] and j > i):
                    suppressed[j] = True
    keep = list(dict.fromkeys(keep))
    return boxes[keep], scores[keep]

def centroid(b):
    return ((float(b[0])+float(b[2]))/2.0, (float(b[1])+float(b[3]))/2.0)

def validate_face_landmarks(lm, bbox):
    if lm is None or lm.shape != (5, 2): return False
    h = float(bbox[3]) - float(bbox[1])
    if h < 14.0: return False
    re, le, nose, rm, lmc = lm
    if not (re[1] < nose[1] and le[1] < nose[1]): return False
    if not (nose[1] < rm[1] and nose[1] < lmc[1]): return False
    eye_dx, eye_dy = abs(le[0]-re[0]), abs(le[1]-re[1])
    if eye_dx < 1e-3: return False
    if eye_dy / max(eye_dx, 1e-3) > 0.85: return False
    if abs(lmc[0]-rm[0]) > 1.8 * eye_dx: return False
    if not (min(rm[0], lmc[0]) - 2 <= nose[0] <= max(rm[0], lmc[0]) + 2): return False
    return True


## 1. YOLO person detector

In [ ]:
class YOLOPersonDetector:
    def __init__(self, cfg):
        self.cfg = cfg
        self.model = YOLO(cfg['yolo_model'])
        self.conf = float(cfg['conf'])
        self.device = cfg.get('device', 'cpu')
        self.imgsz = cfg.get('imgsz')
        self.min_w = float(cfg.get('min_box_w', 20))
        self.min_h = float(cfg.get('min_box_h', 30))
        self.ar_min = float(cfg.get('aspect_min', 0.15))
        self.ar_max = float(cfg.get('aspect_max', 6.0))

    def detect(self, frame_bgr):
        kw = {'conf': self.conf, 'classes': [0], 'verbose': False, 'device': self.device}
        if self.imgsz: kw['imgsz'] = self.imgsz
        res = self.model.predict(frame_bgr, **kw)
        if not res or len(res[0].boxes) == 0:
            return np.zeros((0,4), dtype=np.float32), np.zeros((0,), dtype=np.float32)
        b = res[0].boxes.xyxy.cpu().numpy().astype(np.float32)
        s = res[0].boxes.conf.cpu().numpy().astype(np.float32)
        keep = []
        for i, box in enumerate(b):
            w, h = float(box[2]-box[0]), float(box[3]-box[1])
            if w < self.min_w or h < self.min_h: continue
            ar = w / h
            if ar < self.ar_min or ar > self.ar_max: continue
            keep.append(i)
        if not keep:
            return np.zeros((0,4), dtype=np.float32), np.zeros((0,), dtype=np.float32)
        return b[keep], s[keep]


## 2. YuNet face detector (v6 score bug fixed)

In [ ]:
YUNET_URL = 'https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx'

class YuNetFaceDetector:
    def __init__(self, cfg):
        self.cfg = cfg
        path = cfg['model_path']
        if not os.path.exists(path):
            os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
            print(f'[YuNet] downloading to {path} ...')
            urllib.request.urlretrieve(YUNET_URL, path)
        self.det = cv2.FaceDetectorYN.create(
            path, '', (320, 320),
            float(cfg['conf_threshold']), float(cfg['nms_threshold']),
            int(cfg['top_k']))

    def detect(self, frame_bgr):
        h, w = frame_bgr.shape[:2]
        self.det.setInputSize((w, h))
        ok, faces = self.det.detect(frame_bgr)
        if not ok or faces is None: return []
        out = []
        for row in faces:
            x, y, ww, hh = float(row[0]), float(row[1]), float(row[2]), float(row[3])
            score = float(row[14])  # <-- v6 bug fix (was row[4])
            if score < self.cfg['conf_threshold']: continue
            lm = np.array([
                [float(row[4]), float(row[5])],
                [float(row[6]), float(row[7])],
                [float(row[8]), float(row[9])],
                [float(row[10]), float(row[11])],
                [float(row[12]), float(row[13])],
            ], dtype=np.float32)
            xyxy = np.array([x, y, x+ww, y+hh], dtype=np.float32)
            if not validate_face_landmarks(lm, xyxy): continue
            out.append((xyxy, score, lm))
        return out


## 3. Face ViT gender classifier

In [ ]:
class FaceViTGender:
    def __init__(self, cfg):
        self.cfg = cfg
        name = cfg['model']
        self.proc = AutoImageProcessor.from_pretrained(name)
        self.model = AutoModelForImageClassification.from_pretrained(name)
        self.model.eval()
        self.id2label = self.model.config.id2label
        self.female_idx = self._find_idx(cfg['female_label'])
        self.male_idx = self._find_idx(cfg['male_label'])
        self.conf_thresh = float(cfg['conf_threshold'])

    def _find_idx(self, key):
        key = key.lower()
        for k, v in self.id2label.items():
            if key in str(v).lower(): return int(k)
        raise RuntimeError(f'label {key} not in {self.id2label}')

    def classify(self, face_bgr):
        if face_bgr is None or face_bgr.size == 0: return 'unknown', 0.0
        try:
            pil = Image.fromarray(cv2.cvtColor(face_bgr, cv2.COLOR_BGR2RGB))
            inputs = self.proc(images=pil, return_tensors='pt')
            with torch.no_grad():
                logits = self.model(**inputs).logits[0]
            probs = torch.softmax(logits, dim=0)
            idx = int(probs.argmax().item())
            label = 'female' if idx == self.female_idx else 'male' if idx == self.male_idx else 'unknown'
            return label, float(probs[idx].item())
        except Exception:
            return 'unknown', 0.0


## 4. CLIP zero-shot body gender (NO age)

In [ ]:
class CLIPGender:
    def __init__(self, cfg):
        self.cfg = cfg
        name = cfg['model']
        self.model = CLIPModel.from_pretrained(name)
        self.proc = CLIPProcessor.from_pretrained(name)
        self.model.eval()
        self.gender_text = cfg['gender_male_prompts'] + cfg['gender_female_prompts']
        self.gender_text_emb = self._encode_text(self.gender_text)
        self.uncertain_ratio = float(cfg['uncertain_ratio'])

    def _encode_text(self, texts):
        with torch.no_grad():
            t = self.proc(text=texts, return_tensors='pt', padding=True)
            out = self.model.get_text_features(**t)
            emb = out.pooler_output if hasattr(out, 'pooler_output') else out
            emb = emb.detach()
            emb = emb / emb.norm(dim=-1, keepdim=True)
        return emb.detach()

    def _encode_image(self, pil_img):
        with torch.no_grad():
            inp = self.proc(images=pil_img, return_tensors='pt')
            out = self.model.get_image_features(**inp)
            emb = out.pooler_output if hasattr(out, 'pooler_output') else out
            emb = emb.detach()
            emb = emb / emb.norm(dim=-1, keepdim=True)
        return emb.detach()

    def classify_body(self, body_bgr):
        out = {'gender': 'unknown',
               'gender_probs': {'male': 0.5, 'female': 0.5},
               'ratio': 1.0, 'embedding': None}
        if body_bgr is None or body_bgr.size == 0: return out
        try:
            pil = Image.fromarray(cv2.cvtColor(body_bgr, cv2.COLOR_BGR2RGB))
            img_emb = self._encode_image(pil)
            out['embedding'] = img_emb.cpu().numpy().flatten()
            g = (img_emb @ self.gender_text_emb.T).cpu().numpy()[0]
            n_male = len(self.cfg['gender_male_prompts'])
            male_score = float(np.mean(g[:n_male]))
            female_score = float(np.mean(g[n_male:]))
            mn = max(min(male_score, female_score), 1e-6)
            ratio = max(male_score, female_score) / mn
            probs = np.exp([male_score, female_score])
            probs = (probs / probs.sum()).tolist()
            gender = 'male' if male_score >= female_score else 'female'
            out.update({
                'gender': gender,
                'gender_probs': {'male': float(probs[0]), 'female': float(probs[1])},
                'ratio': float(ratio),
            })
        except Exception:
            pass
        return out


## 5. Ensemble (v9.2 'uncertain' branch)

In [ ]:
def ensemble_gender(face_label, face_conf, clip_gender, clip_ratio,
                    uncertain_ratio=1.8, uncertain_face_threshold=0.70):
    clip_confident = clip_ratio >= uncertain_ratio
    face_confident = (face_conf >= uncertain_face_threshold
                      and face_label in ('male', 'female'))
    if clip_confident:
        if face_confident and face_label != clip_gender:
            return face_label
        return clip_gender
    if face_confident:
        return face_label
    return 'uncertain'  # v9.2 fix (was 'male' in v9)


## 6. Tracker + ReIDBank

In [ ]:
@dataclass
class Track:
    track_id: int
    box: np.ndarray
    last_seen: int
    missed: int = 0
    gender: str = 'uncertain'
    clip_embedding: Optional[np.ndarray] = None
    face_label: str = 'unknown'
    face_conf: float = 0.0
    clip_gender: str = 'unknown'
    clip_ratio: float = 0.0
    first_seen: int = 0

class ReIDBank:
    def __init__(self, cosine_threshold=0.75, expiry_seconds=30.0, max_size=200):
        self.cos_threshold = float(cosine_threshold)
        self.expiry_seconds = float(expiry_seconds)
        self.max_size = int(max_size)
        self._bank = []

    def add(self, embedding, track_id, gender, fps, last_seen_frame):
        if embedding is None: return
        self._bank.append({'embedding': embedding.astype(np.float32),
                           'track_id': track_id,
                           'expired_wall_time': time.time(),
                           'gender': gender})
        if len(self._bank) > self.max_size: self._bank.pop(0)

    def query(self, embedding):
        if embedding is None or not self._bank: return None
        emb = embedding.astype(np.float32)
        emb = emb / max(np.linalg.norm(emb), 1e-6)
        now = time.time()
        best_id, best_sim = None, 0.0
        for rec in self._bank:
            if now - rec['expired_wall_time'] > self.expiry_seconds: continue
            e = rec['embedding']
            e = e / max(np.linalg.norm(e), 1e-6)
            sim = float(np.dot(emb, e))
            if sim > best_sim: best_sim, best_id = sim, rec['track_id']
        if best_id is not None and best_sim >= self.cos_threshold:
            self._bank = [r for r in self._bank if r['track_id'] != best_id]
            return best_id
        return None

def _centroid_dist(a, b):
    ax, ay = centroid(a); bx, by = centroid(b)
    return math.hypot(ax - bx, ay - by)

def _size_ratio(a, b):
    aa = (float(a[2])-float(a[0])) * (float(a[3])-float(a[1]))
    ab = (float(b[2])-float(b[0])) * (float(b[3])-float(b[1]))
    if ab <= 0: return 0.0
    return min(aa, ab) / max(aa, ab)

class Tracker:
    def __init__(self, cfg, reid=None):
        self.iou_thresh = float(cfg['iou_threshold'])
        self.centroid_max = float(cfg['centroid_max_dist'])
        self.persist = int(cfg['persist_frames'])
        self.size_scale = float(cfg['size_scale'])
        self.reid = reid
        self.tracks = {}
        self.next_id = 1

    def update(self, detections, frame_idx, clip_classifier=None):
        active = list(self.tracks.values())
        assigned, used_tracks = set(), set()
        pairs = []
        for di, det in enumerate(detections):
            for tr in active:
                if tr.track_id in used_tracks: continue
                i = iou(det['box'], tr.box)
                if i >= self.iou_thresh:
                    pairs.append((i, di, tr.track_id)); continue
                cd = _centroid_dist(det['box'], tr.box)
                sf = _size_ratio(det['box'], tr.box)
                if cd <= self.centroid_max and sf >= (1.0 - self.size_scale):
                    pairs.append((i - 0.5*(cd/self.centroid_max), di, tr.track_id))
        pairs.sort(reverse=True)
        for _, di, tid in pairs:
            if di in assigned or tid in used_tracks: continue
            det = detections[di]; tr = self.tracks[tid]
            tr.box = det['box']; tr.last_seen = frame_idx; tr.missed = 0
            if det.get('face_label') is not None:
                tr.face_label = det['face_label']; tr.face_conf = det['face_conf']
            if det.get('clip_gender') is not None:
                tr.clip_gender = det['clip_gender']
                tr.clip_ratio = det['clip_ratio']
                tr.gender = det['gender']
                tr.clip_embedding = det.get('clip_embedding', tr.clip_embedding)
            assigned.add(di); used_tracks.add(tid)
        for di, det in enumerate(detections):
            if di in assigned: continue
            new_id = None
            if self.reid is not None and det.get('clip_embedding') is not None:
                new_id = self.reid.query(det['clip_embedding'])
            if new_id is None:
                new_id = self.next_id; self.next_id += 1
            self.tracks[new_id] = Track(
                track_id=new_id, box=det['box'], last_seen=frame_idx,
                first_seen=frame_idx, gender=det.get('gender', 'uncertain'),
                face_label=det.get('face_label', 'unknown'),
                face_conf=det.get('face_conf', 0.0),
                clip_gender=det.get('clip_gender', 'unknown'),
                clip_ratio=det.get('clip_ratio', 0.0),
                clip_embedding=det.get('clip_embedding'))
            assigned.add(di)
        for tid, tr in list(self.tracks.items()):
            if tr.last_seen < frame_idx:
                tr.missed += 1
                if tr.missed > self.persist:
                    if self.reid is not None and tr.clip_embedding is not None:
                        self.reid.add(tr.clip_embedding, tr.track_id, tr.gender,
                                      fps=1.0, last_seen_frame=tr.last_seen)
                    del self.tracks[tid]
        return self.tracks


## 7. Line counter + time-bucket aggregator

In [ ]:
class LineCounter:
    def __init__(self, line_norm, direction='down'):
        self.line_norm = line_norm
        self.direction = direction
        self.prev_centroids = {}
        self.last_state = {}
        self.entries = 0; self.exits = 0
        self.w = 1; self.h = 1

    def set_resolution(self, w, h):
        self.w = w; self.h = h

    def _line_pixels(self):
        (x1, y1), (x2, y2) = self.line_norm
        return ((x1*self.w, y1*self.h), (x2*self.w, y2*self.h))

    def _side(self, p):
        (lx1, ly1), (lx2, ly2) = self._line_pixels()
        cross = (lx2-lx1) * (p[1]-ly1) - (ly2-ly1) * (p[0]-lx1)
        return 1 if cross > 0 else (-1 if cross < 0 else 0)

    def update(self, tracks, frame_idx):
        events = []
        for tid, tr in tracks.items():
            cur = centroid(tr.box)
            prev = self.prev_centroids.get(tid)
            self.prev_centroids[tid] = cur
            if prev is None: continue
            s_prev, s_cur = self._side(prev), self._side(cur)
            if s_prev == 0 or s_cur == 0 or s_prev == s_cur: continue
            direction = 'down' if s_cur > s_prev else 'up'
            last = self.last_state.get(tid)
            self.last_state[tid] = s_cur
            if last == s_cur: continue
            if direction == self.direction:
                self.entries += 1
                events.append({'frame': frame_idx, 'track_id': tid, 'type': 'entry'})
            else:
                self.exits += 1
                events.append({'frame': frame_idx, 'track_id': tid, 'type': 'exit'})
        return events

class TimeBucketAggregator:
    def __init__(self, bucket_seconds=900):
        self.bucket_seconds = int(bucket_seconds)
        self.buckets = defaultdict(lambda: {
            'entries': 0, 'exits': 0,
            'male': 0, 'female': 0, 'uncertain': 0,
            'unique_tracks': set()})

    def add_event(self, t_sec, event, track):
        b = int(t_sec // self.bucket_seconds) * self.bucket_seconds
        rec = self.buckets[b]
        if event['type'] == 'entry': rec['entries'] += 1
        else: rec['exits'] += 1
        if track is not None:
            rec['unique_tracks'].add(track.track_id)
            if track.gender in ('male', 'female', 'uncertain'):
                rec[track.gender] += 1

    def finalise(self):
        out = []
        for b in sorted(self.buckets.keys()):
            rec = self.buckets[b]
            out.append({
                'bucket_start': b,
                'entries': rec['entries'], 'exits': rec['exits'],
                'male': rec['male'], 'female': rec['female'], 'uncertain': rec['uncertain'],
                'unique_tracks': len(rec['unique_tracks']),
            })
        return out


## 8. Annotate + Reports

In [ ]:
COLORS = {'male': (200, 80, 40), 'female': (40, 80, 220), 'uncertain': (120, 120, 120)}

def annotate_frame(frame_bgr, tracks):
    out = frame_bgr.copy()
    for tid, tr in tracks.items():
        x1, y1, x2, y2 = [int(v) for v in tr.box]
        col = COLORS.get(tr.gender, (120, 120, 120))
        cv2.rectangle(out, (x1, y1), (x2, y2), col, 2)
        label = f'#{tid} {tr.gender}'
        if tr.face_conf > 0: label += f' face={tr.face_conf:.2f}'
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        cv2.rectangle(out, (x1, y1-th-6), (x1+tw+4, y1), col, -1)
        cv2.putText(out, label, (x1+2, y1-4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)
    return out

def write_demographics_pie(out_dir, male, female, uncertain):
    import matplotlib.pyplot as plt
    labels = ['Male', 'Female', 'Uncertain']
    sizes = [male, female, uncertain]
    kept_l = [l for l, s in zip(labels, sizes) if s > 0]
    kept_s = [s for s in sizes if s > 0]
    if not kept_s: return None
    fig, ax = plt.subplots(figsize=(5, 5), constrained_layout=True)
    ax.pie(kept_s, labels=kept_l, autopct='%1.1f%%',
           colors=['#4C78A8', '#E45756', '#9AA0A6'], startangle=90)
    ax.set_title('Gender Demographics')
    p = os.path.join(out_dir, 'demographics_pie.png')
    fig.savefig(p); plt.close(fig)
    return p

def write_footfall_png(out_dir, buckets):
    import matplotlib.pyplot as plt
    if not buckets: return None
    x = [b['bucket_start'] / 3600.0 for b in buckets]
    e = [b['entries'] for b in buckets]
    x2 = [b['exits'] for b in buckets]
    fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
    ax.bar([v-0.1 for v in x], e, width=0.2, label='entries', color='#4C78A8')
    ax.bar([v+0.1 for v in x], x2, width=0.2, label='exits', color='#E45756')
    ax.set_xlabel('Hour'); ax.set_ylabel('Count')
    ax.set_title('Footfall by Hour'); ax.legend()
    p = os.path.join(out_dir, 'footfall_by_hour.png')
    fig.savefig(p); plt.close(fig)
    return p


## 9. Pipeline orchestrator

In [ ]:
class StoreAnalytics:
    def __init__(self, cfg):
        self.cfg = cfg
        print('[init] YOLO...'); self.detector = YOLOPersonDetector(cfg['detector'])
        print('[init] YuNet...'); self.face_det = YuNetFaceDetector(cfg['face_detector'])
        print('[init] FaceViT...'); self.face_gender = FaceViTGender(cfg['face_gender'])
        print('[init] CLIP...'); self.clip = CLIPGender(cfg['clip'])
        self.reid = None
        if cfg['reid'].get('enabled'):
            r = {k: v for k, v in cfg['reid'].items() if k != 'enabled'}
            self.reid = ReIDBank(**r)
        self.tracker = Tracker(cfg['tracker'], reid=self.reid)
        self.line_counter = LineCounter(cfg['footfall']['line'], cfg['footfall']['direction'])
        self.agg = TimeBucketAggregator(cfg['aggregator']['bucket_seconds'])
        self.uncertain_ratio = float(cfg['clip']['uncertain_ratio'])
        self.uncertain_face_thresh = float(cfg['clip']['uncertain_face_threshold'])
        self._gender_locked = set()
        self._seen_tracks = {}

    def process_video(self, video_path, out_dir,
                      dump_every=30, process_every_n=1,
                      max_frames=None, start_frame=1, end_frame=None):
        cap = cv2.VideoCapture(video_path)
        assert cap.isOpened(), f'cannot open {video_path}'
        fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if end_frame is None: end_frame = n_total
        if max_frames is not None: end_frame = min(end_frame, start_frame + max_frames - 1)
        self.line_counter.set_resolution(w, h)
        os.makedirs(out_dir, exist_ok=True)
        dump_dir = os.path.join(out_dir, 'annotated_frames')
        os.makedirs(dump_dir, exist_ok=True)
        print(f'[run] {video_path} {w}x{h} @ {fps:.2f}fps frames {start_frame}..{end_frame}')
        if start_frame > 1: cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame - 1)
        frame_idx = start_frame - 1
        processed = 0
        annotated_paths = []
        t0 = time.time()
        while True:
            ok, frame = cap.read()
            if not ok: break
            frame_idx += 1
            if frame_idx > end_frame: break
            if frame_idx % process_every_n != 0: continue
            boxes, scores = self.detector.detect(frame)
            if len(boxes) > 0:
                boxes, scores = dedup_boxes(boxes, scores, 0.45, 0.85)
            faces = self.face_det.detect(frame)
            dets = []
            for bi, box in enumerate(boxes):
                x1, y1, x2, y2 = [int(v) for v in box]
                pad = 20
                cx1, cy1 = max(0, x1-pad), max(0, y1-pad)
                cx2, cy2 = min(w, x2+pad), min(h, y2+pad)
                body = frame[cy1:cy2, cx1:cx2].copy()
                clip_out = self.clip.classify_body(body)
                face_label, face_conf = 'unknown', 0.0
                best_area, best_crop = 0, None
                for (fbox, fscore, flm) in faces:
                    fx, fy = (fbox[0]+fbox[2])/2, (fbox[1]+fbox[3])/2
                    if x1 <= fx <= x2 and y1 <= fy <= y2:
                        fa = (fbox[2]-fbox[0]) * (fbox[3]-fbox[1])
                        if fa > best_area:
                            best_area = fa
                            best_crop = frame[int(fbox[1]):int(fbox[3]),
                                              int(fbox[0]):int(fbox[2])].copy()
                if best_crop is not None and best_crop.size > 0:
                    face_label, face_conf = self.face_gender.classify(best_crop)
                gender = ensemble_gender(face_label, face_conf,
                                          clip_out['gender'], clip_out['ratio'],
                                          uncertain_ratio=self.uncertain_ratio,
                                          uncertain_face_threshold=self.uncertain_face_thresh)
                dets.append({'box': box.astype(np.float32), 'score': float(scores[bi]),
                             'face_label': face_label, 'face_conf': face_conf,
                             'clip_gender': clip_out['gender'], 'clip_ratio': clip_out['ratio'],
                             'gender': gender,
                             'clip_embedding': clip_out['embedding']})
            tracks = self.tracker.update(dets, frame_idx, clip_classifier=self.clip)
            for tid, tr in tracks.items():
                if tid in self._gender_locked:
                    tr.gender = self._seen_tracks[tid]['gender']
                    continue
                if tr.gender in ('male', 'female'):
                    self._gender_locked.add(tid)
                self._seen_tracks[tid] = {'gender': tr.gender}
            events = self.line_counter.update(tracks, frame_idx)
            t_sec = frame_idx / fps
            for ev in events:
                self.agg.add_event(t_sec, ev, tracks.get(ev['track_id']))
            if dump_every > 0 and frame_idx % dump_every == 0:
                ann = annotate_frame(frame, tracks)
                p = os.path.join(dump_dir, f'frame_{frame_idx:06d}.jpg')
                cv2.imwrite(p, ann, [cv2.IMWRITE_JPEG_QUALITY, 85])
                annotated_paths.append(p)
            processed += 1
            if processed % 10 == 0:
                eta = (time.time() - t0) / processed * (end_frame - frame_idx) / process_every_n
                print(f'[run] frame {frame_idx}/{end_frame} '
                      f'tracks={len(tracks)} entries={self.line_counter.entries} '
                      f'exits={self.line_counter.exits} eta={eta:.0f}s')
        cap.release()
        buckets = self.agg.finalise()
        gender_counts = {'male': 0, 'female': 0, 'uncertain': 0}
        for tid, info in self._seen_tracks.items():
            g = info['gender'] if info['gender'] in gender_counts else 'uncertain'
            gender_counts[g] += 1
        # write reports
        import csv
        with open(os.path.join(out_dir, 'daily_report.csv'), 'w', newline='') as fh:
            w = csv.writer(fh)
            w.writerow(['bucket_start','entries','exits','male','female','uncertain','unique_tracks'])
            for b in buckets:
                w.writerow([b['bucket_start'],b['entries'],b['exits'],
                            b['male'],b['female'],b['uncertain'],b['unique_tracks']])
        with open(os.path.join(out_dir, 'per_person.csv'), 'w', newline='') as fh:
            w = csv.writer(fh)
            w.writerow(['track_id','gender'])
            for tid, info in sorted(self._seen_tracks.items()):
                w.writerow([tid, info['gender']])
        pie = write_demographics_pie(out_dir, gender_counts['male'], gender_counts['female'], gender_counts['uncertain'])
        foot = write_footfall_png(out_dir, buckets)
        summary = {
            'video': video_path, 'fps': fps, 'frames_processed': processed,
            'entries': self.line_counter.entries, 'exits': self.line_counter.exits,
            'unique_track_estimate': len(self._seen_tracks),
            'gender_counts': gender_counts, 'buckets': buckets,
            'demographics_pie': pie, 'footfall_by_hour': foot,
            'annotated_frame_samples': annotated_paths[:10],
        }
        with open(os.path.join(out_dir, 'summary.json'), 'w') as fh:
            json.dump(summary, fh, indent=2, default=str)
        return summary


## 10. Upload your video

In [ ]:
# Colab: use the upload widget
try:
    from google.colab import files
    uploaded = files.upload()
    VIDEO_PATH = list(uploaded.keys())[0]
    print('Uploaded:', VIDEO_PATH)
except ImportError:
    # Local Jupyter: just set the path
    VIDEO_PATH = 'input.mp4'  # <-- edit this
    print('Local mode. Set VIDEO_PATH =', VIDEO_PATH)

OUT_DIR = 'outputs/run1'
os.makedirs(OUT_DIR, exist_ok=True)


## 11. Run the pipeline

In [ ]:
sa = StoreAnalytics(CONFIG)
summary = sa.process_video(
    VIDEO_PATH, OUT_DIR,
    dump_every=CONFIG['pipeline']['dump_annotated_every'],
    process_every_n=CONFIG['pipeline']['process_every_n_frames'],
)
print('\n=== SUMMARY ===')
print(json.dumps({
    'frames_processed': summary['frames_processed'],
    'entries': summary['entries'],
    'exits': summary['exits'],
    'unique_track_estimate': summary['unique_track_estimate'],
    'gender_counts': summary['gender_counts'],
}, indent=2))


## 12. View results

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt

# Show demographics pie
if summary.get('demographics_pie') and os.path.exists(summary['demographics_pie']):
    print('Gender demographics:')
    display(Image(filename=summary['demographics_pie']))

# Show footfall chart
if summary.get('footfall_by_hour') and os.path.exists(summary['footfall_by_hour']):
    print('Footfall by hour:')
    display(Image(filename=summary['footfall_by_hour']))

# Show a few annotated frames
print('Sample annotated frames:')
for p in summary['annotated_frame_samples'][:4]:
    if os.path.exists(p):
        display(Image(filename=p, width=600))


## 13. Download outputs (Colab)

In [ ]:
try:
    from google.colab import files
    # Zip the run dir and download
    !cd {OUT_DIR} && zip -r ../run1.zip . > /dev/null
    files.download(f'{OUT_DIR}/../run1.zip')
    print('Downloaded run1.zip')
except ImportError:
    print(f'Local mode. Outputs are in: {OUT_DIR}')
    print('Contents:')
    for root, dirs, files_ in os.walk(OUT_DIR):
        for f in files_:
            print(' ', os.path.join(root, f))


## Done

**What this notebook does NOT do** (intentionally):
- Age classification — will be added in a later version.

**Key knobs to tune** (in the Config cell):
- `clip.uncertain_ratio` — lower = stricter (more 'uncertain' labels)
- `reid.cosine_threshold` — lower = more willing to re-link IDs across gaps
- `footfall.line` — normalised `[x,y],[x,y]` of the crossing line
- `pipeline.process_every_n_frames` — 3 = 3x speedup, ~no accuracy loss

For the modular `src/` version of this code, see the `IPD/src/` package in the repo.
